In [2]:
import sys
sys.path.insert(0, '../src')

from perishable_demand_forecasting.features import build_features
import pandas as pd

# Test on a small slice
df = pd.read_parquet('../data/processed/protein_grid.parquet')
sample = df[df["store_nbr"].isin([1, 2])].copy()

out = build_features(sample)
print("Input columns: ", len(sample.columns))
print("Output columns:", len(out.columns))
print("\nGenerated features:", [c for c in out.columns if c not in sample.columns][:10])
print("\nlag_1 nulls:", out["lag_1"].isna().sum(), "(should equal number of store-item pairs)")

Input columns:  11
Output columns: 34

Generated features: ['dayofweek', 'day', 'month', 'year', 'weekofyear', 'is_weekend', 'is_payday', 'days_in_month', 'lag_1', 'lag_7']

lag_1 nulls: 425 (should equal number of store-item pairs)


In [3]:
n_pairs = sample.groupby(["store_nbr", "item_nbr"], observed=True).ngroups
print("Store-item pairs in sample:", n_pairs)
print("lag_1 nulls:", out["lag_1"].isna().sum())
print("Match:", n_pairs == out["lag_1"].isna().sum())

Store-item pairs in sample: 425
lag_1 nulls: 425
Match: True


In [4]:
import importlib
import perishable_demand_forecasting.predict as P
importlib.reload(P)

model = P.load_model()
print("Model loaded. Features expected:", len(model.feature_name()))

# Real rows from the test set
test = pd.read_parquet('../data/processed/protein_features_hol.parquet')
test = test[test["date"] >= "2017-07-31"].head(100).copy()

preds = P.predict(model, test)
preds_biased = P.predict(model, test, apply_cost_bias=True)

print(f"\nPredictions:        {preds[:5].round(3)}")
print(f"Cost-biased (×0.8): {preds_biased[:5].round(3)}")
print(f"Actual:             {test['unit_sales'].head().values.round(3)}")
print(f"\nRatio check: {(preds_biased[0] / preds[0]):.2f} (should be 0.80)")

Model loaded. Features expected: 43

Predictions:        [1.321 1.206 0.886 0.964 1.893]
Cost-biased (×0.8): [1.056 0.964 0.709 0.771 1.514]
Actual:             [3. 1. 3. 2. 1.]

Ratio check: 0.80 (should be 0.80)


In [5]:
try:
    P.predict(model, test.drop(columns=["lag_7"]))
except ValueError as e:
    print("Caught correctly:", e)

Caught correctly: Missing required features: ['lag_7']
